# PROCESAMIENTO Y NORMALIZACIÓN

In [ ]:
import pandas as pd
import numpy as np
import os
import ipaddress
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN Y CONSTANTES
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define la ruta a tu espacio de trabajo y tu archivo de entrada
BASE_DIR = " "
INPUT_FILE = os.path.join(BASE_DIR, "tu_archivo_de_flujos_raw.csv") 
OUTPUT_DIR = os.path.join(BASE_DIR, "normalizacion")

# Parámetros de procesamiento
CHUNK_SIZE = 500_000
TIME_FREQ = "5min"
MIN_EPSILON = 1e-9

# [CONFIGURACIÓN DE USUARIO]: Ajusta los índices de columnas según la estructura de tu CSV
COL_SRC_IP = 0
COL_TIMESTAMP = 4
COL_BYTES = 33
COL_ETIQUETA = 153

# [CONFIGURACIÓN DE USUARIO]: Define los rangos CIDR de tu red para etiquetar el tráfico.
# Sustituye estas IPs de ejemplo por los rangos reales que desees analizar.
RANGOS_ETIQUETAS = {
    "etiqueta_usuarios_wifi": [ipaddress.ip_network("192.XXX.X.X/XX")],
    "etiqueta_servidores_1":  [ipaddress.ip_network("10.X.X.X/XX")],
    "etiqueta_servidores_2":  [ipaddress.ip_network("172.XX.X.X/XX")]
}

# Validación rápida
RANGOS_VALIDOS = [r for lista in RANGOS_ETIQUETAS.values() for r in lista]

# ==========================================
# 2. FUNCIONES
# ==========================================

def ip_valida(ip_str):
    """Verifica si la IP pertenece a alguno de los rangos definidos."""
    try:
        ip = ipaddress.ip_address(ip_str)
        return any(ip in red for red in RANGOS_VALIDOS)
    except:
        return False

def procesar_todo_unificado(): 
    print("=== INICIANDO PROCESO UNIFICADO: GENERACIÓN + NORMALIZACIÓN ===")
    
    # Preparar entorno
    if not os.path.exists(INPUT_FILE):
        print(f"[ERROR] No se encuentra el archivo de entrada: {INPUT_FILE}")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ------------------------------------------------------
    # FASE 1: LECTURA, FILTRADO Y AGREGACIÓN
    # ------------------------------------------------------
    print(f"[INFO] Leyendo y procesando {INPUT_FILE} en bloques...")
    
    data_accum = []
    
    try:
        reader = pd.read_csv(
            INPUT_FILE, 
            sep=r"\s+", 
            header=None, 
            chunksize=CHUNK_SIZE, 
            engine="python"
        )

        for i, chunk in enumerate(tqdm(reader, desc="Leyendo chunks")):
            # 1. Seleccionar columnas
            sub = chunk[[COL_SRC_IP, COL_TIMESTAMP, COL_BYTES, COL_ETIQUETA]].copy()
            sub.columns = ["src_ip", "timestamp", "bytes", "etiqueta"]

            # 2. Filtrar por IP válida
            sub = sub[sub["src_ip"].apply(ip_valida)]

            if sub.empty:
                continue

            # 3. Conversión Temporal
            sub["timestamp"] = pd.to_datetime(sub["timestamp"], unit="s", errors="coerce")
            sub = sub.dropna(subset=["timestamp"])
            sub["intervalo"] = sub["timestamp"].dt.floor(TIME_FREQ)

            # 4. Agregación Parcial
            grouped = sub.groupby(["etiqueta", "src_ip", "intervalo"])["bytes"].sum().reset_index()
            data_accum.append(grouped)

    except Exception as e:
        print(f"\n[ERROR] Fallo durante la lectura: {e}")
        return

    print("\n[INFO] Consolidando datos en memoria...")
    
    # Unir todos los fragmentos
    if not data_accum:
        print("[WARN] No se encontraron datos válidos en los rangos especificados.")
        return

    df_total = pd.concat(data_accum, ignore_index=True)
    
    # Re-agrupar final
    df_total = df_total.groupby(["etiqueta", "src_ip", "intervalo"])["bytes"].sum().reset_index()

    # ------------------------------------------------------
    # FASE 2: NORMALIZACIÓN Y GUARDADO
    # ------------------------------------------------------
    etiquetas_unicas = df_total["etiqueta"].unique()
    print(f"[INFO] Etiquetas a procesar: {etiquetas_unicas}")
    
    resumen_final = []

    for etiqueta in etiquetas_unicas:
        print(f"\n[PROCESANDO] Generando series para: {etiqueta}")
        
        # Filtrar datos de la etiqueta
        df_label = df_total[df_total["etiqueta"] == etiqueta]
        
        # Pivotar (Time x IPs)
        pivot = df_label.pivot_table(
            index="intervalo", 
            columns="src_ip", 
            values="bytes", 
            aggfunc="sum"
        )
        
        # Sincronización Temporal (Zero-padding)
        start_time = pivot.index.min()
        end_time = pivot.index.max()
        full_range = pd.date_range(start=start_time, end=end_time, freq=TIME_FREQ)
        
        # Reindexar y rellenar ceros
        pivot = pivot.reindex(full_range).fillna(0)
        
        # Normalización Z-Score
        means = pivot.mean()
        stds = pivot.std()
        pivot_norm = (pivot - means) / (stds + MIN_EPSILON)
        pivot_norm = pivot_norm.fillna(0)

        # Guardar CSV final
        out_path = os.path.join(OUTPUT_DIR, f"{etiqueta}.csv")
        pivot_norm.to_csv(out_path)
        
        # Datos para resumen
        duracion = end_time - start_time if pd.notnull(end_time) else "N/A"
        resumen_final.append({
            "Etiqueta": etiqueta,
            "IPs": pivot_norm.shape[1],
            "Muestras": pivot_norm.shape[0],
            "Inicio": start_time,
            "Fin": end_time,
            "Duracion": duracion
        })

    # ------------------------------------------------------
    # FASE 3: INFORME FINAL
    # ------------------------------------------------------
    print("\n" + "="*80)
    print("RESUMEN DE RESULTADOS DE NORMALIZACIÓN")
    print("="*80)
    
    if resumen_final:
        df_res = pd.DataFrame(resumen_final)
        print(df_res.to_string(index=False))
        print(f"\n[INFO] Archivos guardados en: {OUTPUT_DIR}")
    else:
        print("[WARN] No se generaron archivos.")

if __name__ == "__main__":
    procesar_todo_unificado()